# Polyp detection in colonoscopy: does it survive a new hospital?
Run top to bottom. **First:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi -L   # should print a Tesla T4. If it errors, you're not on a GPU runtime.

## 1. Upload the project
Upload `polyp-detection.zip` with the folder icon on the left (or run the cell and pick the file).

In [ ]:
from google.colab import files
import os
if not os.path.exists('polyp-detection.zip'):
    files.upload()
!unzip -oq polyp-detection.zip -d /content/
%cd /content/polyp-detection
!pip install -q -r requirements.txt

## 2. Download the datasets
- **Kvasir-SEG** (1000 images, Norway): https://datasets.simula.no/kvasir-seg/
- **CVC-ClinicDB** (612 frames, Barcelona): file mirror on Hugging Face (official page: https://polyp.grand-challenge.org/CVCClinicDB/)

In [ ]:
!mkdir -p raw
!wget -q --show-progress -O raw/kvasir-seg.zip https://datasets.simula.no/downloads/kvasir-seg.zip
!unzip -oq raw/kvasir-seg.zip -d raw/ || echo '>>> Kvasir download failed: download kvasir-seg.zip manually from https://datasets.simula.no/kvasir-seg/ , upload it into the raw/ folder, then rerun this cell'

from huggingface_hub import snapshot_download
snapshot_download(repo_id='berkaytrhn/cvc-clinicdb', repo_type='dataset',
                  local_dir='raw/cvc', allow_patterns=['CVC-ClinicDB/**'])
!ls raw raw/cvc/CVC-ClinicDB

## 3. Convert masks → YOLO boxes

In [ ]:
!python prepare_data.py --kvasir raw/Kvasir-SEG --cvc raw/cvc/CVC-ClinicDB

## 4. Train the three experiments
Each takes roughly 20–40 min on a T4. Colab can disconnect, so save `runs/` to Drive after each one (cell below).

In [ ]:
!python train.py --exp baseline

In [ ]:
!python train.py --exp strongaug

In [ ]:
!python train.py --exp finetune

In [ ]:
# Optional: back up trained models to Google Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/polyp-detection && cp -r runs /content/drive/MyDrive/polyp-detection/

## 5. Evaluate: same hospital vs new hospital

In [ ]:
!python evaluate.py
from IPython.display import Image, display
display(Image('results/domain_shift.png'))
display(Image('results/cvc_examples_baseline.jpg'))
display(Image('results/cvc_examples_finetune.jpg'))

## 6. Real-time demo video

In [ ]:
!python demo.py --weights runs/finetune/weights/best.pt --source data/yolo/cvc/images/test

## 7. Download everything for GitHub

In [ ]:
!zip -rq results.zip results runs/*/weights/best.pt runs/*/results.png
files.download('results.zip')